In [2]:
# ----------------
# Dependencies
# ----------------

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import DataLoader, TensorDataset

import pandas as pd
import numpy as np
import random

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    average_precision_score,
    f1_score
)

# ------------------
# Reproducibility
# ------------------

def set_seed(seed: int):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    torch.use_deterministic_algorithms(True, warn_only=True)


# ------------------------------------
# Device Setup
# ------------------------------------

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# -----------------------------------
# Load Dataset from GDrive (Colab)
# -----------------------------------

from google.colab import drive
drive.mount('/content/drive')

# ----------------
# Load Dataset
# ----------------

df = pd.read_csv(
    "/content/drive/MyDrive/data/dataset_original.csv"
)

if "hash" in df.columns:
    df = df.drop(columns=["hash"])

X = df.drop(columns=["malware"]).values.astype(np.float32)
y = df["malware"].values.astype(np.int64)

# ------------------------------------------------
# Fixed train/test split (same for every seed)
# ------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

# ------------------------------------
# Hyperparameters
# ------------------------------------

VOCAB_SIZE = 307
SEQ_LEN = 100

EMB_DIM = 128
HIDDEN_DIM = 256

NUM_CLASSES = 2

BATCH_SIZE = 128
EPOCHS = 100

SEEDS = [10, 20, 30, 40, 50]

Using device: cuda
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Train shape: (30713, 100)
Test shape : (13163, 100)


In [3]:
class LSTMClassifier(nn.Module):

    def __init__(self):

        super().__init__()

        self.embedding = nn.Embedding(
            VOCAB_SIZE,
            EMB_DIM
        )

        self.lstm = nn.LSTM(
            input_size=EMB_DIM,
            hidden_size=HIDDEN_DIM,
            num_layers=2,
            dropout=0.3,
            batch_first=True
        )

        self.fc = nn.Sequential(
            nn.Linear(HIDDEN_DIM, 128),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.4),
            nn.Linear(128, NUM_CLASSES)
        )

    def forward(self, x):
        x = x.long()
        x = self.embedding(x)
        output, (hidden, cell) = self.lstm(x)
        x = hidden[-1]
        x = self.fc(x)
        return x

In [ ]:
all_results = []

for seed in SEEDS:

    print("\n" + "=" * 80)
    print(f"TRAINING SEED {seed}")
    print("=" * 80)

    set_seed(seed)

    # -------------------
    # Tensor Conversion
    # -------------------

    X_train_t = torch.tensor(
        X_train,
        dtype=torch.float32
    ).to(DEVICE)

    y_train_t = torch.tensor(
        y_train,
        dtype=torch.long
    ).to(DEVICE)

    X_test_t = torch.tensor(
        X_test,
        dtype=torch.float32
    ).to(DEVICE)

    y_test_t = torch.tensor(
        y_test,
        dtype=torch.long
    ).to(DEVICE)

    train_loader = DataLoader(
        TensorDataset(
            X_train_t,
            y_train_t
        ),
        batch_size=BATCH_SIZE,
        shuffle=True
    )

    # -------------------
    # Model
    # -------------------

    model = LSTMClassifier().to(DEVICE)

    optimizer = optim.Adam(
        model.parameters(),
        lr=2e-4
    )

    history = {
        "loss": []
    }

    # -------------------
    # Training
    # -------------------

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0

        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            logits = model(batch_x)
            loss = F.cross_entropy(
                logits,
                batch_y
            )

            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)

        history["loss"].append(avg_loss)

        print(
            f"Seed {seed} | "
            f"Epoch {epoch+1:03d} | "
            f"Loss {avg_loss:.4f}"
        )

    # -------------------
    # Save History
    # -------------------

    history_df = pd.DataFrame(history)

    history_df.to_csv(
        f"lstm_history_{seed}.csv",
        index=False
    )

    # -------------------
    # Save Model
    # -------------------

    torch.save(
        model.state_dict(),
        f"lstm_seed_{seed}.pt"
    )

    # -------------------
    # Evaluation
    # -------------------

    model.eval()

    with torch.no_grad():

        logits = model(X_test_t)

        probs = F.softmax(
            logits,
            dim=1
        )

        preds = torch.argmax(
            probs,
            dim=1
        )

    report = classification_report(
        y_test_t.cpu().numpy(),
        preds.cpu().numpy(),
        digits=4
    )

    print(report)

    pr_auc = average_precision_score(
        y_test_t.cpu().numpy(),
        probs[:, 1].cpu().numpy()
    )

    roc_auc = roc_auc_score(
        y_test_t.cpu().numpy(),
        probs[:, 1].cpu().numpy()
    )

    macro_f1 = f1_score(
        y_test_t.cpu().numpy(),
        preds.cpu().numpy(),
        average="macro"
    )

    print("PR-AUC :", pr_auc)
    print("ROC-AUC:", roc_auc)
    print("Macro-F1:", macro_f1)

    all_results.append({
        "seed": seed,
        "macro_f1": macro_f1,
        "pr_auc": pr_auc,
        "roc_auc": roc_auc
    })


TRAINING SEED 10
Seed 10 | Epoch 001 | Loss 0.1410
Seed 10 | Epoch 002 | Loss 0.0771
Seed 10 | Epoch 003 | Loss 0.0653
Seed 10 | Epoch 004 | Loss 0.0550
Seed 10 | Epoch 005 | Loss 0.0516
Seed 10 | Epoch 006 | Loss 0.0429
Seed 10 | Epoch 007 | Loss 0.0379
Seed 10 | Epoch 008 | Loss 0.0395
Seed 10 | Epoch 009 | Loss 0.0317
Seed 10 | Epoch 010 | Loss 0.0296
Seed 10 | Epoch 011 | Loss 0.0362
Seed 10 | Epoch 012 | Loss 0.0378
Seed 10 | Epoch 013 | Loss 0.0300
Seed 10 | Epoch 014 | Loss 0.0268
Seed 10 | Epoch 015 | Loss 0.0227
Seed 10 | Epoch 016 | Loss 0.0212
Seed 10 | Epoch 017 | Loss 0.0219
Seed 10 | Epoch 018 | Loss 0.0187
Seed 10 | Epoch 019 | Loss 0.0182
Seed 10 | Epoch 020 | Loss 0.0167
Seed 10 | Epoch 021 | Loss 0.0148
Seed 10 | Epoch 022 | Loss 0.0165
Seed 10 | Epoch 023 | Loss 0.0179
Seed 10 | Epoch 024 | Loss 0.0148
Seed 10 | Epoch 025 | Loss 0.0134
Seed 10 | Epoch 026 | Loss 0.0182
Seed 10 | Epoch 027 | Loss 0.0128
Seed 10 | Epoch 028 | Loss 0.0110
Seed 10 | Epoch 029 | Loss 0.0

In [ ]:
results_df = pd.DataFrame(all_results)

print("\n")
print(results_df)

print("\nMean Results")
print(results_df.mean(numeric_only=True))

print("\nStd Results")
print(results_df.std(numeric_only=True))

results_df.to_csv(
    "lstm_results_summary.csv",
    index=False
)